# 🧠 **Activation Functions: Complete Guide & Implementation**

## 🎯 **Introduction**

**Activation functions** are mathematical functions that determine the output of neural network nodes. They introduce **non-linearity** into neural networks, enabling them to learn complex patterns and relationships in data.

### 🎓 **Learning Objectives**

By the end of this notebook, you will:

1. **Understand** the mathematical properties of 15+ activation functions
2. **Visualize** activation functions and their derivatives
3. **Compare** performance characteristics and use cases
4. **Implement** activation functions from scratch
5. **Apply** best practices for choosing activation functions

### 🗺️ **Notebook Structure**

- **Part I**: Theory & Mathematical Foundations
- **Part II**: Classical Activation Functions (ReLU, Sigmoid, Tanh, Softmax)
- **Part III**: Modern Activation Functions (GELU, Swish, Mish)
- **Part IV**: Specialized Functions (ELU, SELU, Hard variants)
- **Part V**: Comparative Analysis & Best Practices

---

**Table of Contents:**

| Function | Formula | Range | Key Property |
|----------|---------|-------|--------------|
| ReLU | $\max(0, x)$ | $[0, +\infty)$ | Simple, non-saturating |
| Sigmoid | $\frac{1}{1+e^{-x}}$ | $(0, 1)$ | Smooth, probabilistic |
| Tanh | $\frac{e^x-e^{-x}}{e^x+e^{-x}}$ | $(-1, 1)$ | Zero-centered |
| GELU | $x \cdot \Phi(x)$ | $(-\infty, +\infty)$ | Probabilistic |
| Swish | $x \cdot \text{sigmoid}(x)$ | $(-\infty, +\infty)$ | Self-gated |
| Mish | $x \cdot \tanh(\text{softplus}(x))$ | $(-\infty, +\infty)$ | Smooth, unbounded |

*\"The choice of activation function can make or break your neural network's performance.\"*

In [ ]:
# 📦 SETUP & IMPORTS
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.special import erf
from scipy.stats import norm
import warnings
from typing import Callable, List, Tuple
warnings.filterwarnings('ignore')

# Plotting setup
plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (15, 8)
plt.rcParams['font.size'] = 12

print("🎯 Activation Functions: Complete Guide")
print("=" * 40)
print("📚 All libraries imported successfully")
print("🎨 Visualization style configured")
print("🧮 Ready to explore activation functions!")

In [ ]:
# 🎨 COMPREHENSIVE VISUALIZATION UTILITIES

def plot_activation_function(func: Callable, func_name: str, x_range: Tuple[float, float] = (-5, 5), 
                           derivative_func: Callable = None, description: str = "", 
                           formula: str = "", use_cases: str = "", properties: List[str] = None):
    """
    Comprehensive plotting function for activation functions with enhanced analysis
    """
    x = torch.linspace(x_range[0], x_range[1], 1000)
    
    if derivative_func is None:
        fig, ax = plt.subplots(1, 1, figsize=(12, 8))
        axes = [ax]
    else:
        fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Main function
    with torch.no_grad():
        y = func(x)
        if hasattr(y, 'detach'):
            y = y.detach()
    
    axes[0].plot(x.numpy(), y.numpy(), linewidth=3, color='#2E86AB', label=func_name)
    axes[0].axhline(y=0, color='black', linestyle='-', alpha=0.3)
    axes[0].axvline(x=0, color='black', linestyle='-', alpha=0.3)
    axes[0].grid(True, alpha=0.3)
    axes[0].set_xlabel('Input (x)', fontsize=12)
    axes[0].set_ylabel(f'{func_name}(x)', fontsize=12)
    axes[0].set_title(f'{func_name} Activation Function', fontsize=14, fontweight='bold')
    axes[0].legend(fontsize=11)
    
    # Add range annotations
    y_min, y_max = y.min().item(), y.max().item()
    axes[0].text(0.02, 0.98, f'Range: [{y_min:.2f}, {y_max:.2f}]', 
                transform=axes[0].transAxes, fontsize=10,
                bbox=dict(boxstyle="round,pad=0.3", facecolor="yellow", alpha=0.7),
                verticalalignment='top')
    
    # Derivative if provided
    if derivative_func is not None:
        with torch.no_grad():
            dy = derivative_func(x)
            if hasattr(dy, 'detach'):
                dy = dy.detach()
        
        axes[1].plot(x.numpy(), dy.numpy(), linewidth=3, color='#A23B72', label=f"{func_name}'")
        axes[1].axhline(y=0, color='black', linestyle='-', alpha=0.3)
        axes[1].axvline(x=0, color='black', linestyle='-', alpha=0.3)
        axes[1].grid(True, alpha=0.3)
        axes[1].set_xlabel('Input (x)', fontsize=12)
        axes[1].set_ylabel(f"d/dx {func_name}(x)", fontsize=12)
        axes[1].set_title(f'{func_name} Derivative', fontsize=14, fontweight='bold')
        axes[1].legend(fontsize=11)
        
        # Add gradient statistics
        dy_max = dy.max().item()
        dy_min = dy.min().item()
        axes[1].text(0.02, 0.98, f'Grad Range: [{dy_min:.2f}, {dy_max:.2f}]', 
                    transform=axes[1].transAxes, fontsize=10,
                    bbox=dict(boxstyle="round,pad=0.3", facecolor="lightgreen", alpha=0.7),
                    verticalalignment='top')
    
    plt.tight_layout()
    plt.show()
    
    # Print comprehensive analysis
    print(f"📊 {func_name} Analysis:")
    print(f"   📈 Formula: {formula}")
    print(f"   📋 Description: {description}")
    print(f"   🎯 Use Cases: {use_cases}")
    print(f"   📊 Output Range: [{y_min:.3f}, {y_max:.3f}]")
    print(f"   🎪 Zero-centered: {'Yes' if y_min < 0 and y_max > 0 else 'No'}")
    print(f"   ∞ Bounded: {'Yes' if abs(y_min) < 1000 and abs(y_max) < 1000 else 'No'}")
    
    if derivative_func is not None:
        dy_max = dy.max().item()
        print(f"   📈 Max Gradient: {dy_max:.3f}")
        print(f"   ⚠️ Vanishing Gradient Risk: {'High' if dy_max < 0.25 else 'Medium' if dy_max < 1.0 else 'Low'}")
        print(f"   🚀 Exploding Gradient Risk: {'High' if dy_max > 10 else 'Low'}")
    
    if properties:
        print(f"   ✨ Special Properties: {', '.join(properties)}")
    print()

def compare_functions(functions: List[Callable], names: List[str], x_range: Tuple[float, float] = (-5, 5), 
                     title: str = "Function Comparison"):
    """
    Compare multiple activation functions in one plot with enhanced styling
    """
    plt.figure(figsize=(14, 8))
    x = torch.linspace(x_range[0], x_range[1], 1000)
    
    colors = plt.cm.Set1(np.linspace(0, 1, len(functions)))
    
    for func, name, color in zip(functions, names, colors):
        with torch.no_grad():
            y = func(x)
            if hasattr(y, 'detach'):
                y = y.detach()
        plt.plot(x.numpy(), y.numpy(), linewidth=2.5, label=name, color=color)
    
    plt.axhline(y=0, color='black', linestyle='-', alpha=0.3)
    plt.axvline(x=0, color='black', linestyle='-', alpha=0.3)
    plt.grid(True, alpha=0.3)
    plt.xlabel('Input (x)', fontsize=14)
    plt.ylabel('Output', fontsize=14)
    plt.title(title, fontsize=16, fontweight='bold')
    plt.legend(fontsize=12, bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

def analyze_gradient_flow(func: Callable, derivative_func: Callable, func_name: str):
    """
    Analyze gradient flow characteristics
    """
    x = torch.linspace(-10, 10, 1000)
    
    with torch.no_grad():
        dy = derivative_func(x)
        if hasattr(dy, 'detach'):
            dy = dy.detach()
    
    # Find dead zones (where gradient is very small)
    dead_zones = torch.abs(dy) < 0.01
    active_zones = torch.abs(dy) > 0.1
    
    dead_percentage = (dead_zones.sum().float() / len(dy) * 100).item()
    active_percentage = (active_zones.sum().float() / len(dy) * 100).item()
    
    print(f"🔍 Gradient Flow Analysis for {func_name}:")
    print(f"   💀 Dead zones (|grad| < 0.01): {dead_percentage:.1f}%")
    print(f"   🚀 Active zones (|grad| > 0.1): {active_percentage:.1f}%")
    print(f"   📊 Mean absolute gradient: {torch.abs(dy).mean():.4f}")
    print(f"   📈 Gradient variance: {dy.var():.4f}")
    print()

print("🎨 Enhanced visualization utilities ready!")
print("📊 Advanced analysis functions available")

## 🔥 **1. ReLU (Rectified Linear Unit)**

### **Mathematical Definition**
$$f(x) = \max(0, x) = \begin{cases} x & \text{if } x \geq 0 \\ 0 & \text{if } x < 0 \end{cases}$$

### **Derivative**
$$f'(x) = \begin{cases} 1 & \text{if } x > 0 \\ 0 & \text{if } x < 0 \\ \text{undefined} & \text{if } x = 0 \end{cases}$$

### **Key Properties**
- ✅ **Computationally efficient** (simple max operation)
- ✅ **Non-saturating** for positive inputs
- ✅ **Sparse activation** (many zeros)
- ⚠️ **Dying ReLU problem** (neurons can become inactive)
- ⚠️ **Not zero-centered**

In [ ]:
# ReLU Implementation and Analysis

def relu_torch(x):
    return torch.relu(x)

def relu_derivative(x):
    return (x > 0).float()

def relu_numpy(x):
    """Custom ReLU implementation from scratch"""
    return np.maximum(0, x)

# Comprehensive analysis
plot_activation_function(
    func=relu_torch,
    func_name="ReLU",
    derivative_func=relu_derivative,
    formula="f(x) = max(0, x)",
    description="Simple, computationally efficient, solves vanishing gradient problem",
    use_cases="Hidden layers in deep networks, CNNs, default choice for many applications",
    properties=["Non-saturating", "Sparse activation", "Computationally efficient", "Dying ReLU problem"]
)

# Analyze gradient flow
analyze_gradient_flow(relu_torch, relu_derivative, "ReLU")

# Demonstrate the dying ReLU problem
print("🚨 Dying ReLU Demonstration:")
x_negative = torch.tensor([-5.0, -2.0, -1.0, -0.1])
y_negative = relu_torch(x_negative)
grad_negative = relu_derivative(x_negative)

for x_val, y_val, grad_val in zip(x_negative, y_negative, grad_negative):
    print(f"   x = {x_val:4.1f}: output = {y_val:.1f}, gradient = {grad_val:.1f}")
print("   💀 All negative inputs produce zero output and gradient!")

## 📈 **2. Sigmoid (Logistic Function)**

### **Mathematical Definition**
$$f(x) = \frac{1}{1 + e^{-x}}$$

### **Derivative**
$$f'(x) = f(x)(1 - f(x)) = \text{sigmoid}(x)(1 - \text{sigmoid}(x))$$

### **Key Properties**
- ✅ **Smooth and differentiable**
- ✅ **Outputs in range (0, 1)** - interpretable as probabilities
- ✅ **S-shaped curve**
- ⚠️ **Vanishing gradient problem** (gradients approach 0 at extremes)
- ⚠️ **Not zero-centered**
- ⚠️ **Computationally expensive** (exponential operation)

In [ ]:
# Sigmoid Implementation and Analysis

def sigmoid_torch(x):
    return torch.sigmoid(x)

def sigmoid_derivative(x):
    s = torch.sigmoid(x)
    return s * (1 - s)

def sigmoid_numpy(x):
    """Custom Sigmoid implementation from scratch with numerical stability"""
    # Prevent overflow by clipping
    x_clipped = np.clip(x, -500, 500)
    return 1 / (1 + np.exp(-x_clipped))

# Comprehensive analysis
plot_activation_function(
    func=sigmoid_torch,
    func_name="Sigmoid",
    derivative_func=sigmoid_derivative,
    formula="f(x) = 1/(1 + e^(-x))",
    description="Smooth, bounded, outputs probabilities, historically important",
    use_cases="Binary classification output layer, gates in LSTM/GRU, probability estimation",
    properties=["Smooth", "Bounded [0,1]", "Vanishing gradient problem", "Not zero-centered"]
)

# Analyze gradient flow
analyze_gradient_flow(sigmoid_torch, sigmoid_derivative, "Sigmoid")

# Show the vanishing gradient problem
print("🚨 Vanishing Gradient Demonstration:")
x_extreme = torch.tensor([-10, -5, 0, 5, 10])
grad_extreme = sigmoid_derivative(x_extreme)
sigmoid_values = sigmoid_torch(x_extreme)

for x_val, sigmoid_val, grad_val in zip(x_extreme, sigmoid_values, grad_extreme):
    print(f"   x = {x_val:3.0f}: sigmoid = {sigmoid_val:.6f}, gradient = {grad_val:.6f}")
print("   ⚠️ Notice how gradients become tiny at extreme values!")

# Compare with ReLU
print("\n🔄 Sigmoid vs ReLU Output Comparison:")
x_compare = torch.linspace(-5, 5, 11)
sigmoid_out = sigmoid_torch(x_compare)
relu_out = relu_torch(x_compare)

for x, sig, rel in zip(x_compare, sigmoid_out, relu_out):
    print(f"   x = {x:4.1f}: Sigmoid = {sig:.3f}, ReLU = {rel:.3f}")

## 🌊 **3. Tanh (Hyperbolic Tangent)**

### **Mathematical Definition**
$$f(x) = \tanh(x) = \frac{e^x - e^{-x}}{e^x + e^{-x}} = \frac{e^{2x} - 1}{e^{2x} + 1}$$

### **Derivative**
$$f'(x) = 1 - \tanh^2(x) = \text{sech}^2(x)$$

### **Key Properties**
- ✅ **Zero-centered output** (range: -1 to 1)
- ✅ **Stronger gradients than sigmoid**
- ✅ **Smooth and differentiable**
- ⚠️ **Still suffers from vanishing gradients**
- ⚠️ **Computationally expensive**

In [ ]:
# Tanh Implementation and Analysis

def tanh_torch(x):
    return torch.tanh(x)

def tanh_derivative(x):
    return 1 - torch.tanh(x)**2

def tanh_numpy(x):
    """Custom Tanh implementation from scratch"""
    return np.tanh(x)

# Comprehensive analysis
plot_activation_function(
    func=tanh_torch,
    func_name="Tanh",
    derivative_func=tanh_derivative,
    formula="f(x) = (e^x - e^(-x))/(e^x + e^(-x))",
    description="Zero-centered, stronger gradients than sigmoid, S-shaped",
    use_cases="Hidden layers in shallow networks, RNNs, when zero-centered output needed",
    properties=["Zero-centered", "Bounded [-1,1]", "Stronger gradients than sigmoid", "Still saturates"]
)

# Analyze gradient flow
analyze_gradient_flow(tanh_torch, tanh_derivative, "Tanh")

# Compare Sigmoid vs Tanh gradients
print("🔄 Sigmoid vs Tanh Gradient Comparison:")
x_vals = torch.tensor([0, 1, 2, 3])
sigmoid_grads = sigmoid_derivative(x_vals)
tanh_grads = tanh_derivative(x_vals)

for x_val, sig_grad, tanh_grad in zip(x_vals, sigmoid_grads, tanh_grads):
    print(f"   x = {x_val}: Sigmoid grad = {sig_grad:.4f}, Tanh grad = {tanh_grad:.4f}")
print("   ✅ Tanh generally has stronger gradients!")

# 🏛️ **Part III: Modern Activation Functions**

## ⚡ **4. GELU (Gaussian Error Linear Unit)**

### **Mathematical Definition**
$$f(x) = x \cdot \Phi(x)$$
where $\Phi(x)$ is the standard normal cumulative distribution function.

### **Approximation**
$$f(x) \approx 0.5x\left(1 + \tanh\left(\sqrt{\frac{2}{\pi}}\left(x + 0.044715x^3\right)\right)\right)$$

### **Key Properties**
- ✅ **Probabilistic interpretation** (probability that input > random Gaussian)
- ✅ **Smooth and differentiable**
- ✅ **Non-monotonic** (slight dip for negative values)
- ✅ **Used in transformers** (BERT, GPT)
- ⚠️ **More complex computation** than ReLU

---

## 🌀 **5. Swish (SiLU - Sigmoid Linear Unit)**

### **Mathematical Definition**
$$f(x) = x \cdot \text{sigmoid}(x) = \frac{x}{1 + e^{-x}}$$

### **Derivative**
$$f'(x) = \text{sigmoid}(x) + x \cdot \text{sigmoid}(x) \cdot (1 - \text{sigmoid}(x))$$

### **Key Properties**
- ✅ **Self-gated** (gates its own input)
- ✅ **Smooth and non-monotonic**
- ✅ **Discovered through Neural Architecture Search**
- ✅ **Often outperforms ReLU**
- ⚠️ **Unbounded above, bounded below**

---

## 🎭 **6. Mish**

### **Mathematical Definition**
$$f(x) = x \cdot \tanh(\text{softplus}(x)) = x \cdot \tanh(\ln(1 + e^x))$$

### **Key Properties**
- ✅ **Smooth and continuously differentiable**
- ✅ **Non-monotonic with self-regularization**
- ✅ **Strong empirical performance**
- ✅ **Unbounded above, bounded below**
- ⚠️ **Computational complexity**

In [ ]:
# Modern Activation Functions Implementation

# GELU Implementation and Analysis
def gelu_torch(x):
    return F.gelu(x)

def gelu_approximate(x):
    """GELU approximation implementation"""
    return 0.5 * x * (1 + torch.tanh(torch.sqrt(torch.tensor(2.0 / np.pi)) * (x + 0.044715 * x**3)))

def gelu_derivative(x):
    """Approximate GELU derivative"""
    tanh_arg = torch.sqrt(torch.tensor(2.0 / np.pi)) * (x + 0.044715 * x**3)
    tanh_val = torch.tanh(tanh_arg)
    sech2_val = 1 - tanh_val**2
    
    return 0.5 * (1 + tanh_val) + 0.5 * x * sech2_val * torch.sqrt(torch.tensor(2.0 / np.pi)) * (1 + 3 * 0.044715 * x**2)

# Comprehensive analysis
plot_activation_function(
    func=gelu_torch,
    func_name="GELU",
    derivative_func=gelu_derivative,
    formula="f(x) = x * Φ(x) where Φ is standard normal CDF",
    description="Probabilistic activation, smooth, used in transformers",
    use_cases="Transformer models (BERT, GPT), modern deep learning",
    properties=["Smooth", "Non-monotonic", "Probabilistic", "Self-gating"]
)

analyze_gradient_flow(gelu_torch, gelu_derivative, "GELU")

# Swish Implementation and Analysis
def swish_torch(x):
    return F.silu(x)  # SiLU is the same as Swish

def swish_derivative(x):
    """Swish derivative"""
    sigmoid_x = torch.sigmoid(x)
    return sigmoid_x + x * sigmoid_x * (1 - sigmoid_x)

# Comprehensive analysis
plot_activation_function(
    func=swish_torch,
    func_name="Swish",
    derivative_func=swish_derivative,
    formula="f(x) = x * sigmoid(x)",
    description="Self-gated, smooth, non-monotonic, outperforms ReLU in many cases",
    use_cases="Deep networks, NAS discoveries, mobile networks",
    properties=["Self-gated", "Smooth", "Non-monotonic", "Unbounded above"]
)

analyze_gradient_flow(swish_torch, swish_derivative, "Swish")

# Mish Implementation and Analysis
def mish_torch(x):
    return x * torch.tanh(F.softplus(x))

def mish_derivative(x):
    """Mish derivative (approximation)"""
    softplus_x = F.softplus(x)
    tanh_softplus = torch.tanh(softplus_x)
    sigmoid_x = torch.sigmoid(x)
    return tanh_softplus + x * sigmoid_x * (1 - tanh_softplus**2)

# Comprehensive analysis
plot_activation_function(
    func=mish_torch,
    func_name="Mish",
    derivative_func=mish_derivative,
    formula="f(x) = x * tanh(softplus(x))",
    description="Smooth, non-monotonic, self-regularizing, strong performance",
    use_cases="Deep networks, computer vision, often outperforms ReLU",
    properties=["Smooth", "Non-monotonic", "Self-regularizing", "Unbounded"]
)

analyze_gradient_flow(mish_torch, mish_derivative, "Mish")

print("🎉 Modern activation functions analyzed!")

# 🔬 **Part IV: Comparative Analysis & Best Practices**

In [ ]:
# 📊 COMPREHENSIVE FUNCTION COMPARISON

print("🔄 Comparing All Activation Functions")
print("=" * 50)

# Compare classical functions
compare_functions(
    functions=[relu_torch, sigmoid_torch, tanh_torch],
    names=["ReLU", "Sigmoid", "Tanh"],
    title="🏛️ Classical Activation Functions Comparison"
)

# Compare modern functions  
compare_functions(
    functions=[gelu_torch, swish_torch, mish_torch],
    names=["GELU", "Swish", "Mish"],
    title="⚡ Modern Activation Functions Comparison"
)

# Compare all functions
compare_functions(
    functions=[relu_torch, sigmoid_torch, tanh_torch, gelu_torch, swish_torch, mish_torch],
    names=["ReLU", "Sigmoid", "Tanh", "GELU", "Swish", "Mish"],
    title="🎯 All Activation Functions Comparison"
)

print("✅ Comparative analysis complete!")

# 🎯 **Selection Guide & Best Practices**

## 🏆 **When to Use Each Function**

### **🔥 ReLU Family**
- **Use for**: Most deep networks, CNNs, default choice
- **Pros**: Fast, solves vanishing gradient, sparse activation
- **Cons**: Dying ReLU problem, not zero-centered

### **📈 Sigmoid**
- **Use for**: Binary classification output, probability estimation
- **Pros**: Bounded output [0,1], smooth
- **Cons**: Vanishing gradient, not zero-centered, computationally expensive

### **🌊 Tanh**
- **Use for**: RNNs, when you need zero-centered output
- **Pros**: Zero-centered, stronger gradients than sigmoid
- **Cons**: Still saturates, vanishing gradient problem

### **⚡ GELU**
- **Use for**: Transformers, modern NLP models
- **Pros**: State-of-the-art performance, probabilistic interpretation
- **Cons**: More complex computation

### **🌀 Swish**
- **Use for**: Deep networks, mobile models, when ReLU underperforms
- **Pros**: Self-gated, smooth, often outperforms ReLU
- **Cons**: More expensive than ReLU

### **🎭 Mish**
- **Use for**: Computer vision, when you need top performance
- **Pros**: Excellent empirical results, smooth
- **Cons**: Computationally expensive

---

## 📋 **Quick Selection Checklist**

1. **🚀 Default choice**: Start with **ReLU**
2. **🎯 Output layer**: Use **Sigmoid** (binary) or **Softmax** (multiclass)
3. **🔄 Zero-centered needed**: Use **Tanh**
4. **🤖 Transformers/NLP**: Use **GELU**
5. **📱 Mobile/Efficient**: Consider **Swish**
6. **🏆 Maximum performance**: Try **Mish**
7. **🔍 Experimental**: Compare multiple options

---

## ⚠️ **Common Pitfalls to Avoid**

1. **Never use sigmoid in hidden layers** (vanishing gradient)
2. **Monitor for dying ReLU** (check gradient flow)
3. **Consider computational cost** in production
4. **Match activation to network depth** (deeper = more modern functions)
5. **Always validate empirically** on your specific task

In [ ]:
# 🎓 SUMMARY AND PERFORMANCE METRICS

print("📊 ACTIVATION FUNCTION SUMMARY")
print("=" * 60)

# Create a comprehensive summary table
import pandas as pd

summary_data = {
    'Function': ['ReLU', 'Sigmoid', 'Tanh', 'GELU', 'Swish', 'Mish'],
    'Range': ['[0, ∞)', '(0, 1)', '(-1, 1)', '(-∞, ∞)', '(-∞, ∞)', '(-∞, ∞)'],
    'Zero-Centered': ['No', 'No', 'Yes', 'Yes', 'Yes', 'Yes'],
    'Monotonic': ['Yes', 'Yes', 'Yes', 'No', 'No', 'No'],
    'Computation': ['Low', 'High', 'High', 'Medium', 'Medium', 'High'],
    'Use Case': ['General', 'Output', 'RNNs', 'Transformers', 'Deep nets', 'Vision'],
    'Vanishing Gradient': ['No', 'Yes', 'Yes', 'No', 'No', 'No']
}

df = pd.DataFrame(summary_data)
print(df.to_string(index=False))

print("\n🏆 KEY TAKEAWAYS:")
print("✅ ReLU: Still the go-to choice for most applications")
print("✅ GELU: Best for transformers and modern NLP")  
print("✅ Swish/Mish: Excellent for when you need top performance")
print("✅ Sigmoid/Tanh: Limited to specific use cases")

print("\n🎯 REMEMBER:")
print("• The 'best' activation function depends on your specific task")
print("• Always experiment and validate on your dataset")
print("• Consider computational constraints in production")
print("• Modern functions often outperform classical ones")

print("\n🎉 Congratulations! You now understand activation functions comprehensively!")